# Task 2 — Core Profiling

Runs **after** Task 1 loads a confirmed header (or headerless `-1` with `unnamed_*` cols).

Three exact checks (source of truth for Phase 1 reports):

| Check | Dimension | What it returns |
|---|---|---|
| Missing values | Completeness | count, %, row indices per column |
| Type consistency | Validity | dominant type + mismatched samples |
| Duplicates | Uniqueness | duplicate row indices (keep first) |

### Tool choice
**pandas** (`isna`, type probing, `duplicated`) — exact indices for reports.

### Why not whylogs as source of truth
whylogs is fast for approximate monitoring, but Phase 1 needs **exact row indices** of every issue.  
Decision: pandas = truth; whylogs optional later for summaries only.

### Important (verified on real Easby files)
- A column that is **100% null** (e.g. Discount) is reported as failed completeness — that is **correct**, not a bug; the source column is empty.
- Missing % matches hand-checks (e.g. Unit Net Price ~34.66% on 2024 sheet).
- 0 duplicate rows on clean transactional sheets is expected.

**Output:** a `CheckResult` per column per check (missing values, duplicates, type consistency) with exact counts, percentages, and row indices -- the source-of-truth numbers used in the Phase 1 report.


> **Source of truth:** production logic lives in data_quality_engine/. This notebook keeps a small readable re-implementation for learning; run python main.py for the full pipeline used in demos.


In [ ]:
from dataclasses import dataclass, field
from typing import Any, Sequence

import pandas as pd


@dataclass
class CheckResult:
    check_name: str
    status: str  # passed | failed | error
    column: str | None
    issues_found: int
    details: dict[str, Any] = field(default_factory=dict)
    dimension: str = ""


def check_missing_values(df: pd.DataFrame) -> list[CheckResult]:
    """Null count + % missing per column. Empty strings are NOT treated as null here."""
    try:
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df must be a DataFrame")
        if df.empty:
            return [CheckResult("missing_values", "passed", None, 0, {"reason": "empty"}, "completeness")]

        total = len(df)
        out = []
        for col in df.columns:
            idx = df.index[df[col].isna()].tolist()
            n = len(idx)
            out.append(
                CheckResult(
                    "missing_values",
                    "passed" if n == 0 else "failed",
                    str(col),
                    n,
                    {
                        "missing_count": n,
                        "missing_pct": round((n / total) * 100, 4) if total else 0.0,
                        "row_indices": idx[:100],
                    },
                    "completeness",
                )
            )
        return out
    except Exception as e:
        return [CheckResult("missing_values", "error", None, 0, {"error": str(e)}, "completeness")]


def check_duplicates(df: pd.DataFrame, subset: Sequence[str] | None = None) -> list[CheckResult]:
    """Flag duplicate rows; keep='first' so first occurrence is clean."""
    try:
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df must be a DataFrame")
        if df.empty:
            return [CheckResult("duplicates", "passed", None, 0, {"reason": "empty"}, "uniqueness")]

        mask = df.duplicated(subset=list(subset) if subset else None, keep="first")
        idx = df.index[mask].tolist()
        n = len(idx)
        return [
            CheckResult(
                "duplicates",
                "passed" if n == 0 else "failed",
                ",".join(subset) if subset else None,
                n,
                {
                    "duplicate_count": n,
                    "duplicate_pct": round((n / len(df)) * 100, 4),
                    "total_rows": len(df),
                    "subset": list(subset) if subset else "all_columns",
                    "row_indices": idx[:100],
                },
                "uniqueness",
            )
        ]
    except Exception as e:
        return [CheckResult("duplicates", "error", None, 0, {"error": str(e)}, "uniqueness")]


def _classify(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return "number"
    if isinstance(value, str):
        return None if value.strip() == "" else "string"
    return type(value).__name__


def _looks_numeric(value: Any) -> bool:
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return not pd.isna(value)
    if isinstance(value, str):
        try:
            float(value.strip().replace(",", ""))
            return True
        except ValueError:
            return False
    return False


def check_type_consistency(series: pd.Series) -> CheckResult:
    """Flag values that don't match the dominant non-null type in the column."""
    col = str(series.name) if series.name is not None else None
    try:
        kinds = [k for v in series.tolist() if (k := _classify(v)) is not None]
        if not kinds:
            return CheckResult("type_mismatch", "passed", col, 0, {"reason": "all_null"}, "validity")

        counts: dict[str, int] = {}
        for k in kinds:
            counts[k] = counts.get(k, 0) + 1
        dominant = max(counts, key=counts.get)

        bad_idx, bad_vals = [], []
        for idx, v in series.items():
            k = _classify(v)
            if k is None:
                continue
            ok = _looks_numeric(v) if dominant == "number" else (k == dominant)
            if not ok:
                bad_idx.append(idx)
                bad_vals.append(v)

        n = len(bad_idx)
        return CheckResult(
            "type_mismatch",
            "passed" if n == 0 else "failed",
            col,
            n,
            {
                "dominant_type": dominant,
                "type_counts": counts,
                "mismatch_pct": round((n / len(kinds)) * 100, 4),
                "row_indices": bad_idx[:100],
                "sample_values": [str(v) for v in bad_vals[:20]],
            },
            "validity",
        )
    except Exception as e:
        return CheckResult("type_mismatch", "error", col, 0, {"error": str(e)}, "validity")


def check_type_consistency_frame(df: pd.DataFrame) -> list[CheckResult]:
    return [check_type_consistency(df[c]) for c in df.columns]


def run_core_profiling(df: pd.DataFrame, dup_subset: Sequence[str] | None = None) -> dict[str, list[CheckResult]]:
    return {
        "missing": check_missing_values(df),
        "types": check_type_consistency_frame(df),
        "duplicates": check_duplicates(df, subset=dup_subset),
    }


def profiling_summary(results: dict[str, list[CheckResult]]) -> pd.DataFrame:
    rows = []
    for group, items in results.items():
        for r in items:
            rows.append({
                "group": group,
                "check": r.check_name,
                "dimension": r.dimension,
                "column": r.column,
                "status": r.status,
                "issues": r.issues_found,
                "highlight": (
                    r.details.get("missing_pct")
                    or r.details.get("mismatch_pct")
                    or r.details.get("duplicate_pct")
                    or r.details.get("dominant_type")
                    or r.details.get("reason")
                ),
                "samples": r.details.get("sample_values") or r.details.get("row_indices", [])[:5],
            })
    return pd.DataFrame(rows)


print("Task 2 checks ready")

## Main demo 
One null, one type mismatch (`"thirty"`), one full-row duplicate.

In [ ]:
df = pd.DataFrame({
    "Name": ["Ali", None, "Sara", "Ali", "Ali"],
    "Age": [30, 25, "thirty", 30, 30],
    "City": ["Lahore", "Karachi", "Lahore", "Lahore", "Lahore"],
    "OrderID": ["O1", "O2", "O3", "O1", "O1"],
})
# make last row an exact duplicate of row 3
df.loc[4] = df.loc[3]
display(df)

results = run_core_profiling(df)
summary = profiling_summary(results)
print("\nProfiling summary:")
display(summary)

print("\n--- Missing (detail) ---")
for r in results["missing"]:
    print(
        f"{r.column:10} {r.status:7} count={r.issues_found} "
        f"pct={r.details.get('missing_pct')}% indices={r.details.get('row_indices')}"
    )

print("\n--- Types (detail) ---")
for r in results["types"]:
    print(
        f"{r.column:10} {r.status:7} issues={r.issues_found} "
        f"dominant={r.details.get('dominant_type')} "
        f"counts={r.details.get('type_counts')} samples={r.details.get('sample_values')}"
    )

print("\n--- Duplicates (detail) ---")
dup = results["duplicates"][0]
print(
    f"{dup.status} count={dup.issues_found} pct={dup.details.get('duplicate_pct')}% "
    f"indices={dup.details.get('row_indices')}"
)

## Package check (source of truth)

Same checks via data_quality_engine.engine.checks — this is what main.py runs.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "data_quality_engine").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_quality_engine.engine.checks.missing_values import check_missing_values
from data_quality_engine.engine.checks.duplicates import check_duplicates
from data_quality_engine.engine.checks.type_mismatch import check_type_consistency_frame

missing = check_missing_values(df)
dups = check_duplicates(df)
types = check_type_consistency_frame(df)
print("missing issues:", sum(r.issues_found for r in missing))
print("duplicate rows:", dups[0].issues_found)
print("type-mismatch columns:", sum(1 for r in types if r.issues_found > 0))
print("OK — package Task 2 checks import and run.")


## Subset duplicates (business key)
Full-row duplicate check can miss "same OrderID, different City". Use `subset`.

Also: **100% null columns are real data issues** (completeness), not detector bugs.

In [ ]:
orders = pd.DataFrame({
    "OrderID": ["O1", "O1", "O2", "O3"],
    "City": ["Lahore", "Karachi", "Lahore", "Islamabad"],  # O1 appears twice with different cities
    "Amount": [100, 100, 50, 75],
})
display(orders)

full = check_duplicates(orders)[0]
key = check_duplicates(orders, subset=["OrderID"])[0]

print("Full-row duplicates:", full.status, "issues=", full.issues_found, full.details.get("row_indices"))
print("OrderID-key duplicates:", key.status, "issues=", key.issues_found, key.details.get("row_indices"))
print("(Key check catches the business duplicate that full-row misses.)")

## Edge cases

## 100% null column (real source emptiness)
Matches the real “Discount is entirely empty” case — missing check should fail at 100%.

In [ ]:
df_discount = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Discount": [None, None, None],  # empty in source file
    "Qty": [1, 2, 3],
})
display(df_discount)
for r in check_missing_values(df_discount):
    print(f"{r.column:10} {r.status:7} missing={r.issues_found} pct={r.details.get('missing_pct')}%")
assert check_missing_values(df_discount)[1].details["missing_pct"] == 100.0
print("OK — 100% null column flagged correctly")

In [ ]:
cases = {
    "clean": pd.DataFrame({"A": [1, 2, 3], "B": ["x", "y", "z"]}),
    "all_null_col": pd.DataFrame({"A": [1, 2], "B": [None, None]}),
    "numeric_as_strings": pd.DataFrame({"Qty": ["10", "20", "N/A", "30"]}),
    "empty": pd.DataFrame(),
}

for name, frame in cases.items():
    print(f"\n=== {name} ===")
    if frame.empty and name == "empty":
        display(frame)
    else:
        display(frame)
    res = run_core_profiling(frame)
    display(profiling_summary(res))

## Optional: tiny whylogs snapshot (not source of truth)
Skip / ignore if `whylogs` is not installed — pandas results above still stand.

In [ ]:
demo = pd.DataFrame({
    "Name": ["Ali", None, "Sara"],
    "Age": [30, 25, "thirty"],
})

try:
    import whylogs as why

    profile = why.log(demo).profile()
    view = profile.view()
    print("whylogs profile columns:", list(view.get_columns().keys()))
    print("Useful for approximate monitoring — NOT for exact issue indices.")
    print("Our pandas report still owns row-level findings:")
    display(profiling_summary(run_core_profiling(demo)))
except ImportError:
    print("whylogs not installed — fine for Phase 1. pandas remains the source of truth.")
    display(profiling_summary(run_core_profiling(demo)))

## Quick comparison

| | pandas (default) | whylogs |
|---|---|---|
| Exact row indices | **yes** | no (approx) |
| Missing % | exact | approximate |
| Type mismatches | exact samples | limited |
| Duplicate rows | exact | weak / not primary |
| Speed on huge files | good enough | faster summaries |
| Phase 1 choice | **YES** | optional later |

Same logic is packaged in:
`missing_values.py`, `type_mismatch.py`, `duplicates.py`.

## Where Column Classification Fits in the Pipeline

A new module, `engine/column_classifier.py`, was added to classify every column as `identifier`, `measurement`, `categorical`, `date`, `pii`, or `free_text` (see the Task 3 notebook for why -- outliers must not run on identifier/PII columns).

**Pipeline placement:** in `main.py`, classification now runs as an explicit, printed step **right after Task 1 (header confirmed) and before Task 2**, not buried inside Task 3. Reasoning:
- Task 1 must finish first -- classification needs real column names and typed data, which only exist after the header row is confirmed and the sheet is loaded.
- Task 2's own checks (missing values, duplicates, type consistency) are **role-agnostic** -- they scan every column the same way regardless of whether it's an identifier or a measurement, so classification doesn't change Task 2's logic or results at all.
- Task 3 (outliers) is the check that actually *depends* on the role (no mean/std/IQR on identifier or PII columns), so classification must be available before Task 3 runs -- and since it's cheap and useful context for every downstream report, it's computed once, early, rather than hidden inside Task 3.
- Task 4 (PII) does **not** need this step -- `detect_pii_in_series()` scans and classifies PII-ness per column on its own.

The cell below imports the real package function and classifies the same demo `df` used above, so you can see how each of its columns would be treated once the pipeline reaches Task 3.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_quality_engine.engine.column_classifier import classify_columns

roles = classify_columns(df)
for col, role in roles.items():
    print(f"{col:10} -> {role}")

**Actual output on this tiny 5-row demo:** `Name` and `City` come out `categorical` as expected. `Age` and `OrderID` both come out `free_text`, which is worth explaining rather than hiding:
- `Age` is `[30, 25, "thirty", 30, 30]` -- only 4/5 values parse as numbers (80%), just under the classifier's 90% numeric-ratio threshold for `measurement`. That's the *same* dirty value the Task 2 type-consistency check above already caught -- the classifier is telling you the same thing a different way: this column isn't clean enough to trust with mean/std yet.
- `OrderID` is `["O1","O2","O3","O1","O1"]` -- only 3 unique values in 5 rows (60% cardinality), well under the 90% threshold the classifier requires before treating short repeated-looking codes as an `identifier`. With only 5 rows and heavy repetition, there isn't enough signal to distinguish "identifier column" from "low-cardinality categorical column" -- and `free_text` is the deliberately conservative fallback for exactly this kind of ambiguity.

**Takeaway:** the classifier's thresholds (90% numeric ratio, 90% cardinality ratio) are tuned for realistic sheet sizes (hundreds/thousands of rows), where cardinality and numeric-ness are much stronger signals. On a toy 5-row demo like this one, don't over-read a `free_text` label -- see the Task 3 notebook's 15-row `invoice_no` example for a case with enough rows to classify cleanly as `identifier`. Either way, only `measurement`-labelled columns get real IQR/KNN stats in Task 3, so ambiguous columns like these are safely skipped, never wrongly flagged.